# Worked example: from data to a baseline

**Time:** about 45-60 minutes. **Before you start:** complete Chapters 0-3, open `microbial-ml.Rproj`, and restart R. Open `scripts/worked-example.R`.

**Question:** what error do we get if we predict the log10 mesophilic spore count of each held-out milk sample using only the average from training farms?

This is a complete prediction workflow, even though its model is deliberately simple. It gives later models an honest reference. We have not yet tested whether farm management or weather improves it.

## A. Know the target before touching the data

MSC means **mesophilic spore count**, measured here in CFU/mL. One row in our selected data will be one MSC assay result for one farm and sampling round. A farm can have several rows.

We use the original assay table linked by [Qian et al. (2025)](https://doi.org/10.1016/j.jfp.2025.100477). We exclude records carrying a note and require a positive, finite concentration. This produces 525 records, matching the paper's reported MSC sample size. These are explicit rules for this teaching slice, not a claim to reproduce every step in the paper.

The [data audit](../reference/data-audit.md) explains the source version, exclusions and detection limit. The four-row practice table in Chapter 2 was invented; this table contains real published research data.

## B. Download once and inspect

**Where: RStudio script.** The supplied helper downloads a fixed author-file version, verifies its checksum and caches it. It uses base R; no package installation is needed. Read `scripts/course_setup.R` if you want to inspect the download code. The helper does not fit a model.

In [1]:
source("scripts/course_setup.R")
source_file <- fetch_course_data()
milk <- read.csv(source_file, stringsAsFactors = FALSE, na.strings = c("", "NA"))
dim(milk)
names(milk)
stopifnot(identical(names(milk), c("FarmID", "Sampling", "Test", "Conc", "Notes")))
stopifnot(nrow(milk) == 2736)

[1] 2736    5

[1] "FarmID"   "Sampling" "Test"     "Conc"     "Notes"

The expected shape is **2,736 rows and 5 columns**. `na.strings = c("", "NA")` tells R how missing text is encoded in this file. If the shape or names differ, stop and investigate; do not simply change the assertions to make them pass.

| Column | Meaning and role |
|---|---|
| `FarmID` | Farm identifier; used to keep groups together |
| `Sampling` | Sampling-round label; helps identify a farm-round record |
| `Test` | Assay type; different assays must not be mixed into one outcome |
| `Conc` | The assay-specific concentration; units depend on `Test` |
| `Notes` | Laboratory notes affecting eligibility |

Inspect the assay types:

In [2]:
table(milk$Test)


APC BAB MSC PSC TSC 
563 513 562 536 562 

The whole file is not one homogeneous training table. We first select a specific test.

## C. Select one assay and make quality rules explicit

In [3]:
msc <- milk[milk$Test == "MSC", ]
nrow(msc)
table(msc$Notes, useNA = "ifany")

[1] 562


        lab error other 8-E was 160              TNTC              <NA> 
               34                 1                 2               525 

There are 562 raw MSC records: 525 have no note, 34 are marked laboratory error, two are marked `TNTC` (too numerous to count), and one has another note. A count marked `TNTC` is not an ordinary quantified value. The other note has not been resolved independently, so this pilot excludes it rather than interpreting it silently.

In the next block, `trimws()` removes surrounding spaces from notes and `is.finite()` checks that a numerical value is neither missing nor infinite. `&` means both conditions must hold; `|` means either can hold. The filtering rule is chosen for data quality, before looking at prediction scores.

In [4]:
blank_note <- is.na(msc$Notes) | trimws(msc$Notes) == ""
positive_count <- is.finite(msc$Conc) & msc$Conc > 0
usable <- msc[blank_note & positive_count, ]
rownames(usable) <- NULL
nrow(usable)
length(unique(usable$FarmID))
stopifnot(nrow(usable) == 525)
stopifnot(length(unique(usable$FarmID)) == 102)
stopifnot(anyDuplicated(usable[c("FarmID", "Sampling")]) == 0)

[1] 525

[1] 102

You should get **525 eligible records from 102 farms**. The checks also confirm that a farm-round occurs only once in this selected assay. Excluding flagged records can affect representativeness; retain an audit of what was excluded and why.

## D. Define the outcome and its limits

The paper reports replacing below-detection MSC values by **25% of a 0.5 CFU/mL detection limit**: 0.125 CFU/mL. We preserve that encoded value and flag it. We do not call it a precise observed concentration.

In [5]:
usable$left_censored <- usable$Conc == 0.125
usable$log10_msc <- log10(usable$Conc)
table(usable$left_censored)
stopifnot(sum(usable$left_censored) == 30)
stopifnot(all(is.finite(usable$log10_msc)))


FALSE  TRUE 
  495    30 

Thirty eligible records have this encoded value. The other 495 are above it. The equality flag identifies this source's published encoding; it is not a general method for discovering detection limits in arbitrary data.

We predict `log10_msc`, the base-10 logarithm of the concentration:

In [6]:
log10(c(1, 10, 100))
10^mean(log10(c(10, 100)))
mean(c(10, 100))

[1] 0 1 2

[1] 31.62278

[1] 55

Log10 values 0, 1 and 2 correspond to 1, 10 and 100 CFU/mL. A negative log value can represent a positive concentration below 1 CFU/mL. It is not a negative number of organisms.

Notice that averaging on the log scale and back-transforming gives about 31.62 for 10 and 100, whereas their arithmetic mean is 55. Our model predicts on the log scale; exponentiating its mean does not automatically estimate the mean on the original count scale.

## E. Hold out farms before learning the baseline

We are asking about a sample from a farm not used to learn the predictor. We therefore hold out whole farms. The seed makes the random selection reproducible; sorting provides a stable starting order. `ceiling()` rounds upward, so 20% of 102 farms gives 21 test farms.

`unique()` lists each farm once. `%in%` asks whether a farm belongs to the selected test list, and `!` reverses that answer for the training set. The supplied `RNGkind()` line fixes random-generator settings for reproducibility; you can use it as given without memorizing its arguments.

In [7]:
RNGkind(kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
set.seed(2026)
farm_ids <- sort(unique(usable$FarmID), method = "radix")
test_farms <- sample(farm_ids, size = ceiling(0.20 * length(farm_ids)))
train <- usable[!usable$FarmID %in% test_farms, ]
test <- usable[usable$FarmID %in% test_farms, ]
c(train_rows = nrow(train), test_rows = nrow(test),
  train_farms = length(unique(train$FarmID)),
  test_farms = length(unique(test$FarmID)))

train_rows   test_rows train_farms  test_farms 
        420         105          81          21

The supplied split has 420 training records from 81 farms and 105 test records from 21 farms. The record fraction does not need to be exactly 20%, because the groups differ in size.

Check the actual groups, not just the row counts:

In [8]:
overlap <- intersect(unique(train$FarmID), unique(test$FarmID))
overlap
stopifnot(length(overlap) == 0)
stopifnot(nrow(train) + nrow(test) == nrow(usable))

character(0)

`character(0)` means the intersection is empty. If it contains any farm identifier, fix the split before fitting or reporting error.

```{warning} Why not split rows at random?
Rows from one farm are related. Letting the procedure learn from a farm and then testing it on that same farm answers a different question from generalizing to unseen farms. Grouping by farm addresses this particular overlap; it does not eliminate every source of dependence or guarantee future performance.
```

## F. Learn the simplest baseline

Use **only the training outcomes** to calculate a mean. Every test record receives that same prediction. No weather or management predictors are used in this baseline.

In [9]:
baseline_log10 <- mean(train$log10_msc)
baseline_log10
test$prediction_log10 <- rep(baseline_log10, nrow(test))
stopifnot(all(test$prediction_log10 == mean(train$log10_msc)))

[1] 0.7450425

The prediction is about 0.7450 log10 CFU/mL for every held-out record. Do not calculate the baseline from all 525 outcomes: that would use the test outcomes in the model.

## G. Evaluate on held-out records

A residual here is **observed minus predicted**. RMSE is the square root of the average squared residual; MAE is the average absolute residual. Both are reported on the log10 CFU/mL scale, and smaller values mean less error.

In [10]:
test$error_log10 <- test$log10_msc - test$prediction_log10
rmse <- sqrt(mean(test$error_log10^2))
mae <- mean(abs(test$error_log10))
data.frame(model = "Training-mean baseline", n_test = nrow(test),
           RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae)

model,n_test,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<int>,<dbl>,<dbl>
Training-mean baseline,105,0.5289231,0.409148


These are record-weighted errors on this one held-out set. They are not confidence intervals and do not establish which later model will perform best. This split is used here for a fixed baseline demonstration; later model tuning must use the training portion rather than repeatedly consulting these test results.

## H. Reconstruct the calculations

In [11]:
rmse_check <- sqrt(sum((test$log10_msc - baseline_log10)^2) / nrow(test))
mae_check <- sum(abs(test$log10_msc - baseline_log10)) / nrow(test)
stopifnot(isTRUE(all.equal(rmse, rmse_check)))
stopifnot(isTRUE(all.equal(mae, mae_check)))
round(data.frame(observed = head(test$log10_msc, 3),
                 predicted = head(test$prediction_log10, 3),
                 residual = head(test$error_log10, 3)), 4)

observed,predicted,residual
<dbl>,<dbl>,<dbl>
0.8129,0.745,0.0679
0.5441,0.745,-0.2010
-0.3010,0.745,-1.0461


Use the first displayed row to subtract predicted from observed yourself. The table is rounded for display; the assertions use full precision. The second expressions for RMSE and MAE check the calculation over every test row.

**Ask AI:**

> I split my milk records by FarmID before calculating the training mean of log10 MSC. Write a short base-R check that no farms occur in both train and test, and reconstruct RMSE from the saved observations and predictions. Do not refit the model or change the split. Explain what this check does not establish.

Compare the proposed check with the actual objects and output above. An assistant must not invent a better score or claim that the split proves independence.

## I. Keep the detection-limit caveat visible

The main score includes the encoded below-detection outcomes. As a descriptive sensitivity check, evaluate the **same fixed predictions** on test rows not carrying that flag:

In [12]:
# Evaluation subset only: the baseline is not refitted or chosen using this result.
quantified_test <- test[!test$left_censored, ]
quantified_rmse <- sqrt(mean(quantified_test$error_log10^2))
data.frame(evaluation = c("All eligible test records", "Quantified test records only"),
           n = c(nrow(test), nrow(quantified_test)),
           RMSE_log10_CFU_ml = c(rmse, quantified_rmse))

evaluation,n,RMSE_log10_CFU_ml
<chr>,<int>,<dbl>
All eligible test records,105,0.5289231
Quantified test records only,104,0.5062914


Only one test record has the flag in this split. The second score describes a different subset. It does not estimate error for all unobserved true concentrations, and it does not fix the censoring problem. A later study would examine more suitable censored-data methods and sensitivity analyses.

## J. Save evidence, not just an AI conversation

In [13]:
dir.create("outputs/baseline", recursive = TRUE, showWarnings = FALSE)
score <- data.frame(model = "Training-mean baseline", training_mean_log10 = baseline_log10,
                    train_rows = nrow(train), test_rows = nrow(test),
                    train_farms = length(unique(train$FarmID)), test_farms = length(test_farms),
                    RMSE_log10_CFU_ml = rmse, MAE_log10_CFU_ml = mae,
                    quantified_test_n = nrow(quantified_test), quantified_test_RMSE = quantified_rmse)
write.csv(score, "outputs/baseline/scores.csv", row.names = FALSE)
split_table <- data.frame(FarmID = farm_ids,
                         split = ifelse(farm_ids %in% test_farms, "test", "train"))
write.csv(split_table, "outputs/baseline/farm-split.csv", row.names = FALSE)
write.csv(test[c("FarmID", "Sampling", "left_censored", "log10_msc", "prediction_log10", "error_log10")],
          "outputs/baseline/test-predictions.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "outputs/baseline/session-info.txt")
file.exists("outputs/baseline/scores.csv")

[1] TRUE

Open **Files → outputs → baseline**. You should see:

- `scores.csv`: the fitted training mean and evaluation metrics;
- `farm-split.csv`: the saved group assignment;
- `test-predictions.csv`: the held-out observations, predictions and residuals;
- `session-info.txt`: the R environment used for this run.

Create a plot using the training data:

In [14]:
png("outputs/baseline/training-distribution.png", width = 1200, height = 720, res = 150)
par(mar = c(4.5, 4.5, 2.5, 1), family = "sans", col.axis = "#294943", col.lab = "#294943")
hist(train$log10_msc, breaks = 18, col = "#8cc7b6", border = "white",
     main = "The baseline is learned from training farms",
     xlab = "Training log10 MSC (CFU/mL)", ylab = "Number of records")
abline(v = baseline_log10, col = "#a94c31", lwd = 3)
legend("topright", "Training mean", col = "#a94c31", lwd = 3, bty = "n")
invisible(dev.off())
cat("Saved outputs/baseline/training-distribution.png\n")

Saved outputs/baseline/training-distribution.png


![Distribution of training log10 MSC with the training mean marked by a vertical line.](../outputs/baseline/training-distribution.png)

Export the selection and verification counts:

In [15]:
audit <- data.frame(check = c("source rows", "raw MSC rows", "excluded lab-error rows",
  "excluded TNTC rows", "excluded other-note rows", "eligible MSC rows",
  "farms", "left-censored encoded rows", "missing eligible counts",
  "duplicate eligible farm-round keys", "train/test farm overlap"),
  value = c(nrow(milk), nrow(msc), sum(msc$Notes == "lab error", na.rm = TRUE),
  sum(msc$Notes == "TNTC", na.rm = TRUE), sum(msc$Notes == "other 8-E was 160", na.rm = TRUE),
  nrow(usable), length(farm_ids), sum(usable$left_censored), sum(is.na(usable$Conc)),
  anyDuplicated(usable[c("FarmID", "Sampling")]), length(overlap)))
write.csv(audit, "outputs/baseline/data-audit.csv", row.names = FALSE)
cat("All data and baseline checks passed.\n")

All data and baseline checks passed.


## K. Write a restrained result

> Using the prespecified farm-level split, a training-mean baseline predicted held-out log10 MSC with RMSE about 0.529 and MAE about 0.409 log10 CFU/mL. The model used 420 training records and was evaluated on 105 records from 21 other farms. It used no management or weather predictors. Its score depends on this split and the treatment of below-detection values. The result does not establish causality or product safety.

The figures above come from executed code. When your input, split or target changes, regenerate the results and update the text. Do not reuse a paragraph containing stale numbers.

```{admonition} Final checkpoint
:class: tip
Restart R and run `scripts/worked-example.R` from top to bottom. Show the zero-overlap result, reconstruct one residual, locate the saved score, and explain why the baseline mean came from training data only.
```

**Transfer exercise:** in a separate copy, use a different seed. Before running, predict which outputs can change and which scientific rules must remain true. Record the original and revised settings; do not pick a seed because it produces a preferred score.

[Download the full R script](../scripts/worked-example.R). The [downloads page](../reference/downloads.md) also contains the complete student project.